# Global Deforestation Analysis (1990–2025)
## Notebook 05: Country Rankings, Deforestation Hotspots & Continental Disparities

### Research Objectives
Deforestation is not evenly distributed across the globe; it is intensely concentrated in specific tropical forest frontiers. This notebook conducts:
1. Identification and ranking of the **Top 10 Most Deforested Countries** between 1990 and 2025.
2. Identification of the **Top 10 Forest Recovery Leaders** (Afforestation & Reforestation).
3. Hotspot detection across major rainforest basins:
   - **The Amazon Basin & Cerrado** (Brazil, Bolivia, Peru, Colombia)
   - **The Congo Basin** (DRC, Cameroon, Gabon, Congo)
   - **Southeast Asia** (Indonesia, Malaysia, Myanmar)
4. Continental aggregation and multi-decadal trends.


In [1]:
import os, sys
import pandas as pd
import numpy as np
import plotly.graph_objects as go
import plotly.express as px

PROJ_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
sys.path.append(PROJ_ROOT)
from src.utils import load_processed_data, THEME, format_metric
from src.charts import create_top_countries_bar

data = load_processed_data()
df_hotspots = data["hotspots"]
df_panel = data["country_panel"]
print("Hotspots and Country panel loaded successfully.")


Hotspots and Country panel loaded successfully.


### 1. Top 10 Countries with Highest Cumulative Forest Loss (1990–2025)
Ranking sovereign countries by cumulative net forest loss in million hectares.


In [2]:
fig_top_loss = create_top_countries_bar(df_hotspots, top_n=10, ascending=True)
fig_top_loss.show()


### 2. Top 10 Countries with Highest Net Forest Gain (1990–2025)
Countries that have added the largest areas of forest through afforestation and natural regeneration.


In [3]:
fig_top_gain = create_top_countries_bar(df_hotspots, top_n=10, ascending=False)
fig_top_gain.show()


### 3. Hotspot Data Table & Percentage Depletion
Countries experiencing the highest percentage loss of their total forest estate since 1990.


In [4]:
top_loss_table = df_hotspots.sort_values("Net_Change_1990_to_Latest_Mha").head(10)[[
    "ISO3", "Entity", "Continent", "Forest_Area_1990_ha", "Forest_Area_Latest_ha", "Net_Change_1990_to_Latest_Mha", "Pct_Change_Since_1990", "Hotspot_Status"
]]
top_loss_table["Forest_Area_1990_Mha"] = (top_loss_table["Forest_Area_1990_ha"] / 1e6).round(2)
top_loss_table["Forest_Area_Latest_Mha"] = (top_loss_table["Forest_Area_Latest_ha"] / 1e6).round(2)
top_loss_table[["Entity", "Continent", "Forest_Area_1990_Mha", "Forest_Area_Latest_Mha", "Net_Change_1990_to_Latest_Mha", "Pct_Change_Since_1990", "Hotspot_Status"]]


,Entity,Continent,Forest_Area_1990_Mha,Forest_Area_Latest_Mha,Net_Change_1990_to_Latest_Mha,Pct_Change_Since_1990,Hotspot_Status
0,Brazil,South America,618.44,486.09,-132.35664,-21.401582,Severe Hotspot (Critical)
1,Democratic Republic of Congo,Africa,160.54,139.19,-21.35116,-13.299572,Severe Hotspot (Critical)
2,Indonesia,Asia,116.33,95.97,-20.36520,-17.505727,Severe Hotspot (Critical)
3,Tanzania,Africa,58.16,43.40,-14.76500,-25.384682,Severe Hotspot (Critical)
4,Angola,Africa,77.53,63.26,-14.27096,-18.406236,Severe Hotspot (Critical)
5,Myanmar,Asia,39.22,27.10,-12.12316,-30.911856,Severe Hotspot (Critical)
6,Paraguay,South America,25.68,14.30,-11.38229,-44.324112,Severe Hotspot (Critical)
7,Peru,South America,77.04,67.16,-9.87813,-12.822431,High Deforestation Hotspot
8,Argentina,South America,55.99,46.60,-9.39162,-16.773859,High Deforestation Hotspot
9,Mozambique,Africa,41.59,32.24,-9.34601,-22.472361,High Deforestation Hotspot


### 4. Continental Aggregation (1990–2025)
Aggregating national forest stocks and net annual rates across all six continents directly from the master panel.


In [5]:
cont_panel = df_panel[df_panel["Continent"].isin(["Africa", "Asia", "Europe", "North America", "South America", "Oceania"])]
cont_agg = cont_panel.groupby(["Continent", "Year"]).agg({
    "Forest_Area_Mha": "sum",
    "Annual_Change_Mha": "sum"
}).reset_index()

fig_cont = px.line(
    cont_agg,
    x="Year",
    y="Forest_Area_Mha",
    color="Continent",
    title="Continental Total Forest Area Trajectories (1990–2025)",
    labels={"Forest_Area_Mha": "Forest Area (Million Hectares)"}
)
fig_cont.update_layout(template="plotly_white")
fig_cont.show()


### Statistical Findings & Hotspot Conclusions
1. **Brazil as the Epicenter**: Brazil has experienced the world's highest absolute forest loss, losing over **132 Million hectares** of forest since 1990, followed by the Democratic Republic of Congo and Indonesia.
2. **African Accelerated Loss**: Sub-Saharan Africa (DRC, Tanzania, Angola, Mozambique) is currently the continent with the highest net annual forest loss rate, driven by population growth, subsistence farming, and charcoal energy demand.
3. **Severe Relative Depletion**: Countries such as Paraguay and Myanmar have lost substantial percentages (20% to over 35%) of their total native forest cover in only 35 years.
